# Clinical Mortality Predictor for Heart Failure Patients
This notebook builds an end-to-end Machine Learning pipeline to predict patient mortality (`DEATH_EVENT`) using clinical records. It is designed to be executed directly in **Google Colab**.

### Pipeline Stages:
1. **Data Collection & Ingestion**
2. **Data Preprocessing & Cleaning**
3. **Feature Engineering**
4. **Data Splitting**
5. **Model Training & Tuning**
6. **Model Evaluation**
7. **Deployment & Monitoring**

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib

## 1. Data Collection & Ingestion
Gathering raw data from various sources (databases, APIs, sensors).

In this section, we ingest the dataset directly from a public raw GitHub URL. This ensures reproducibility and ease of running in Google Colab.

In [ ]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/dimikara/Survival-Prediction-of-Patients-with-Heart-Failure/master/heart_failure_clinical_records_dataset.csv"

print(f"Ingesting data from: {DATA_URL}")
df = pd.read_csv(DATA_URL)
print(f"Dataset successfully loaded. Shape: {df.shape}")

In [ ]:
df.head()

## 2. Data Preprocessing & Cleaning
Handling missing values, removing duplicates, and normalizing data to make it usable.

In [ ]:
print("--- Missing Values per Column ---")
print(df.isnull().sum())

duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

print("\n--- Column Data Types ---")
print(df.dtypes)

In [ ]:
numeric_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium']

print("Outlier Detection (Values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]):")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f" - {col}: {len(outliers)} outliers (range: [{df[col].min()}, {df[col].max()}])")

> **Pre-processing Note**: The dataset does not contain missing values or duplicate rows. Outliers (e.g. extreme values of Creatinine Phosphokinase and Serum Creatinine) are common in clinical records due to patients with acute kidney failure or severe cardiac stress. Rather than removing these vital clinical signals, we will keep them and leverage robust algorithms (like Decision Trees/Random Forests) and robust scaling methods.

## 3. Feature Engineering
Selecting and creating the most relevant features to improve model performance.

### Avoid Data Leakage:
The `time` variable in the dataset represents the patient's **follow-up duration (days)**. Because survival duration is only known *after* the patient dies or completes the study, it cannot be used at baseline to predict whether a patient will die. We **must drop** the `time` feature to prevent severe data leakage.

### Clinical Feature Creation:
1. **`creatinine_sodium_ratio`**: Creatinine and Sodium levels are correlated with cardiac and renal dysfunction. High creatinine coupled with low sodium is a classical indicator of poor prognosis.
2. **`is_elderly`**: Patients aged 65 or above are clinically classified in the high-risk elderly demographic.

In [ ]:
df_engineered = df.copy()

df_engineered['creatinine_sodium_ratio'] = df_engineered['serum_creatinine'] / df_engineered['serum_sodium']

df_engineered['is_elderly'] = (df_engineered['age'] >= 65).astype(int)

df_engineered = df_engineered.drop(columns=['time'])

print(f"Features after engineering (excluding 'time'):\n{list(df_engineered.columns)}")
df_engineered.head()

## 4. Data Splitting
Dividing data into training, validation, and testing sets.

Because the dataset has a moderate class imbalance (~68% survival, ~32% mortality), we perform a **stratified split** to ensure training and test sets maintain identical label proportions.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_engineered.drop(columns=['DEATH_EVENT'])
y = df_engineered['DEATH_EVENT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"\nTrain class distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest class distribution:\n{y_test.value_counts(normalize=True)}")

## 5. Model Training & Tuning
Choosing algorithms and optimizing hyperparameters (using tools like Grid Search).

We build predictive classifiers using two distinct algorithms:
1. **Random Forest Classifier** (robust tree ensemble)
2. **XGBoost Classifier** (highly efficient gradient boosted trees)

We set up hyperparameter tuning via **GridSearchCV** with 5-fold cross-validation, optimizing for **ROC AUC** to handle class imbalance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

rf_param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [3, 5, 8, None],
    'rf__min_samples_split': [2, 5, 10]
}

print("Tuning Random Forest...")
rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print(f"Best Random Forest Parameters: {rf_grid.best_params_}")
print(f"Best Random Forest CV ROC AUC: {rf_grid.best_score_:.4f}")

In [ ]:
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=ratio))
])

xgb_param_grid = {
    'xgb__n_estimators': [50, 100, 150],
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgb__max_depth': [3, 4, 5, 6]
}

print("Tuning XGBoost...")
xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train, y_train)

print(f"Best XGBoost Parameters: {xgb_grid.best_params_}")
print(f"Best XGBoost CV ROC AUC: {xgb_grid.best_score_:.4f}")

## 6. Model Evaluation
Validating the model's accuracy, precision, and recall against testing data.

We compare both tuned models on the held-out test set using ROC AUC, confusion matrices, and detailed classification reports.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt

best_rf = rf_grid.best_estimator_
best_xgb = xgb_grid.best_estimator_

models = {
    "Tuned Random Forest": best_rf,
    "Tuned XGBoost": best_xgb
}

for name, model in models.items():
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    print("=" * 50)
    print(f"EVALUATING: {name}")
    print("=" * 50)
    print(f"Test ROC AUC: {roc_auc_score(y_test, probs):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, preds))
    print("\n")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ConfusionMatrixDisplay.from_estimator(best_rf, X_test, y_test, ax=axes[0, 0], cmap='Blues')
axes[0, 0].set_title("Random Forest Confusion Matrix")

ConfusionMatrixDisplay.from_estimator(best_xgb, X_test, y_test, ax=axes[0, 1], cmap='Purples')
axes[0, 1].set_title("XGBoost Confusion Matrix")

RocCurveDisplay.from_estimator(best_rf, X_test, y_test, ax=axes[1, 0])
axes[1, 0].plot([0, 1], [0, 1], 'r--')
axes[1, 0].set_title("Random Forest ROC Curve")

RocCurveDisplay.from_estimator(best_xgb, X_test, y_test, ax=axes[1, 1])
axes[1, 1].plot([0, 1], [0, 1], 'r--')
axes[1, 1].set_title("XGBoost ROC Curve")

plt.tight_layout()
plt.show()

In [ ]:
xgb_model = best_xgb.named_steps['xgb']
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("XGBoost Feature Importances (Clinical Covariates Only)", fontsize=14, fontweight='bold')
plt.bar(range(X.shape[1]), importances[indices], align="center", color='purple', alpha=0.7)
plt.xticks(range(X.shape[1]), [X.columns[i] for i in indices], rotation=45, ha='right')
plt.ylabel("Relative Importance")
plt.tight_layout()
plt.show()

## 7. Deployment & Monitoring
Deploying the model to production (using Flask, FastAPI, or cloud services) and monitoring for performance decay (model drift).

### 7.1. Model Export
First, we serialize our complete preprocessing and classification pipeline.

In [ ]:
import joblib

model_filename = 'heart_failure_xgb_pipeline.joblib'
joblib.dump(best_xgb, model_filename)
print(f"Serialized pipeline saved to: {model_filename}")

### 7.3. Model Drift & Production Monitoring

Once a clinical predictive model is deployed in a production setting, its performance will degrade over time due to shifts in patient populations or clinical standards. This is referred to as **Model Drift**.

#### Two Main Types of Drift:
1. **Covariate Shift (Data Drift)**: The distribution of input features ($P(X)$) changes over time. For example, a hospital might begin treating older patients on average, shifting the distribution of the `age` feature.
2. **Concept Drift**: The relationship between inputs and the target label ($P(Y|X)$) changes. For example, a new drug or treatment standard might drastically decrease mortality rate for patients who have high serum creatinine, rendering the original model's predictions obsolete.

#### Implementing a Monitoring Strategy:
- **Data Quality Auditing**: Monitor input statistical distributions (Min, Max, Mean, Variance) weekly. Use statistical tests like the **Kolmogorov-Smirnov (KS) test** or **Population Stability Index (PSI)** to compare production data against training benchmarks.
- **Metric Logging**: Log patient inputs, predicted risk probabilities, and actual clinical outcomes (ground truth labels) when they become available. Monitor sliding window accuracy, ROC AUC, and recall.
- **Alerting**: Set up automatic alerts if PSI exceeds 0.2 (indicating significant drift) or if model test ROC AUC drops below 0.70.
- **Triggered Retraining**: Schedule retraining cycles when drift is detected. Collect new labeled patient records, append them to the training set, and execute hyperparameter re-tuning.


---

# Project B: Survival Modeling & Risk Profiling (Survival Analysis)

Unlike classification (where we predict *whether* an event will happen and must drop `time` to prevent leakage), **Survival Analysis** models *when* the event will happen. It accounts for **censoring** (patients who did not die during the follow-up period).

In this section, we build a survival analysis pipeline using the **Cox Proportional Hazards (CoxPH)** model to predict individual patient survival curves over time.

In [ ]:
!pip install -q lifelines

### B.1. Setup & Data Preparation
For survival analysis, our target includes both the duration (`time`) and the event indicator (`DEATH_EVENT`).

In [ ]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

df_survival = pd.read_csv(DATA_URL)

df_survival['creatinine_sodium_ratio'] = df_survival['serum_creatinine'] / df_survival['serum_sodium']
df_survival['is_elderly'] = (df_survival['age'] >= 65).astype(int)

print(f"Survival dataset shape: {df_survival.shape}")

### B.2. Train / Test Split for Survival Data
We split our data into training (80%) and testing (20%) sets. We stratify by `DEATH_EVENT` to ensure equal proportion of event occurrences in both splits.

In [ ]:
train_df, test_df = train_test_split(
    df_survival, test_size=0.2, random_state=42, stratify=df_survival['DEATH_EVENT']
)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

### B.3. Training Cox Proportional Hazards Model
We fit the CoxPH model on the training dataset. We specify `time` as our duration column and `DEATH_EVENT` as our event column.

In [ ]:
cph = CoxPHFitter(penalizer=0.1)
cph.fit(train_df, duration_col='time', event_col='DEATH_EVENT')

cph.print_summary()

### B.4. Model Evaluation (Concordance Index)
The **Concordance Index (C-index)** measures the model's ability to correctly order survival times. A C-index of 0.5 indicates random predictions, while 1.0 indicates perfect ordering.

In [ ]:
train_c_index = concordance_index(
    train_df['time'], 
    -cph.predict_partial_hazard(train_df), 
    train_df['DEATH_EVENT']
)

test_c_index = concordance_index(
    test_df['time'], 
    -cph.predict_partial_hazard(test_df), 
    test_df['DEATH_EVENT']
)

print(f"Train Concordance Index: {train_c_index:.4f}")
print(f"Test Concordance Index: {test_c_index:.4f}")

### B.5. Individual Patient Risk Profiling
One of the major strengths of survival analysis is predicting **individualized survival curves** over time.

Let's select two distinct patients from our test set:
1. **Patient A**: Younger patient with standard ejection fraction and serum creatinine levels.
2. **Patient B**: Elderly patient with low ejection fraction and high serum creatinine levels.

We will compare their predicted survival probabilities over the 250+ day monitoring timeline.

In [ ]:
low_risk_candidates = test_df[
    (test_df['serum_creatinine'] < 1.0) & 
    (test_df['ejection_fraction'] > 45) & 
    (test_df['age'] < 55)
]

high_risk_candidates = test_df[
    (test_df['serum_creatinine'] > 1.8) & 
    (test_df['ejection_fraction'] < 30) & 
    (test_df['age'] > 70)
]

print(f"Low-risk candidates found: {len(low_risk_candidates)}")
print(f"High-risk candidates found: {len(high_risk_candidates)}")

In [ ]:
patient_low = low_risk_candidates.iloc[[0]].drop(columns=['time', 'DEATH_EVENT'])
patient_high = high_risk_candidates.iloc[[0]].drop(columns=['time', 'DEATH_EVENT'])

print("--- Low-Risk Patient Profile ---")
print(patient_low.to_string(index=False))

print("\n--- High-Risk Patient Profile ---")
print(patient_high.to_string(index=False))

In [ ]:
surv_low = cph.predict_survival_function(patient_low)
surv_high = cph.predict_survival_function(patient_high)

plt.figure(figsize=(10, 6))
plt.plot(surv_low.index, surv_low.values, label='Patient A: Low Risk Profile', color='green', linewidth=2.5)
plt.plot(surv_high.index, surv_high.values, label='Patient B: High Risk Profile', color='red', linewidth=2.5)

plt.title('Predicted Individual Patient Survival Probabilities Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Time (Days)', fontsize=12)
plt.ylabel('Survival Probability', fontsize=12)
plt.ylim(0, 1.05)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
days = [30, 90, 180]
print("Predicted Survival Probabilities Comparison:")
print("-" * 55)
for day in days:
    idx_low = surv_low.index.get_indexer([day], method='nearest')[0]
    idx_high = surv_high.index.get_indexer([day], method='nearest')[0]
    
    prob_low = surv_low.iloc[idx_low].values[0]
    prob_high = surv_high.iloc[idx_high].values[0]
    
    print(f"{day}-Day Survival Prob: Patient A = {prob_low:.2%}, Patient B = {prob_high:.2%}")


---

# Project C: Model Interpretability with SHAP

Clinical models must be transparent. We use **SHAP (SHapley Additive exPlanations)** to explain individual patient risk predictions from the trained XGBoost model. This reveals which clinical features drove a patient's risk up or down.

In [ ]:
!pip install -q shap

### C.1. Compute SHAP Values
We extract the trained XGBoost model from the pipeline and use a TreeExplainer to calculate SHAP values for the test set.

In [ ]:
import shap

xgb_model = best_xgb.named_steps['xgb']
scaler = best_xgb.named_steps['scaler']

X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test_scaled)

print("SHAP values calculated successfully. Shape:", shap_values.shape)

### C.2. Global Feature Attribution (Summary Plot)
The summary plot displays how high or low values of a feature impact the mortality prediction across all test patients.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, show=False)
plt.title("SHAP Global Feature Importance & Directional Impact", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### C.3. Local Patient Attribution (Waterfall Plot)
We generate a waterfall plot for the first patient in the test set to explain why they were classified as high or standard risk.

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[0], show=False)
plt.title("Patient-Specific SHAP Risk Attribution", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---

# Project D: Patient Phenotyping via Unsupervised Clustering

We use unsupervised learning to discover patient subgroups (phenogroups) based on continuous clinical features. This helps identify patient cohorts with distinct medical profiles and risk profiles.

### D.1. Standardize and Cluster Patients
We extract all continuous features, scale them, and apply **K-Means clustering** to partition patients into 3 clinical groups.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

continuous_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium']
X_continuous = df_survival[continuous_cols]

scaler_cluster = StandardScaler()
X_continuous_scaled = scaler_cluster.fit_transform(X_continuous)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_continuous_scaled)

df_clusters = df_survival.copy()
df_clusters['Cluster'] = cluster_labels

print("Cluster assignments count:")
print(df_clusters['Cluster'].value_counts())

### D.2. Visualizing Clusters via PCA
We project our continuous features into 2D space using Principal Component Analysis (PCA) to visually inspect cluster separation.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_continuous_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1], 
    hue=df_clusters['Cluster'], 
    palette='Set1', alpha=0.8, s=80
)
plt.title("Patient Clustering Visualized via PCA", fontsize=14, fontweight='bold')
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### D.3. Cluster Profiling
We group the dataset by cluster labels and compute mean features to clinically define each patient phenotype and check their actual mortality rates.

In [ ]:
profile_cols = continuous_cols + ['DEATH_EVENT']
cluster_profile = df_clusters.groupby('Cluster')[profile_cols].mean()
print(cluster_profile.to_string())